# Audio8-TTS-Realistic-Air — multilingual zero-shot voice clone on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/audio8-tts-realistic-air-e052/colab/Audio8-TTS-Realistic-Air-voice-clone.ipynb)

[`Edge0/Audio8-TTS-Realistic-Air`](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air) is Edge0's **0.6B** multilingual text-to-speech model (Apache-2.0). It is a DualAR transformer (Fish Audio S2 Pro–style): a slow AR predicts one semantic token per frame, a fast AR fills 10 codec codebooks. The Hub repo ships the **601M** main checkpoint, tokenizer, processor, custom Transformers code, and a bundled **44.1 kHz** neural codec (`codec.pth`). No extra codec download.

**23 languages.** Preview's 11 (Cantonese, Chinese, Dutch, English, French, German, Italian, Japanese, Korean, Polish, Spanish) plus Arabic, Finnish, Hindi, Indonesian, Malay, Norwegian, Portuguese, Russian, Swedish, Thai, Turkish, and Vietnamese. Quality varies by language — see the [card's evaluation tables](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air#evaluation).

This is the free-Colab multilingual zero-shot cloning notebook. [Silencio-2](Silencio-2-tts-voice-clone.ipynb) is tiny English with tags. [Phonon 31M](phonon-31m-en-streaming-tts.ipynb) is tinier English CPU TTS. [Swar](Swar-voice-clone-tts.ipynb) is Nepali/English F5-TTS. [Maya 2 Native](Maya2-Native-indic-tts.ipynb) is 2.50B Indic + English. [Ming-3.0-tts 0.5B](Ming-3.0-tts-0.5B.ipynb) is a different 0.5B stack. [Drama 3](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

Inference is stock Transformers: `AutoProcessor` + `AutoModel` with **`trust_remote_code=True`**. Same API as the card (and as Preview). This notebook does **not** install the [SGLang Omni](https://github.com/Edge0-AI/Audio8_TTS/tree/master/sglang_omni) serving adapter.

**Licence:** [Apache-2.0](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air/blob/main/LICENSE) on code and weights. See [NOTICE](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air/blob/main/NOTICE).

**Consent:** clone a voice only with its owner's permission. Do not impersonate people or deceive listeners. The clone demo below reuses a clip **this notebook just synthesised**. Label synthetic speech as synthetic where it could be mistaken for a person.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. A free T4 has ~15–16 GB and **no native bfloat16**. The card's CUDA default is `torch.bfloat16` (smoke-tested on an H20). This notebook loads **float16** on compute capability &lt; 8.0 and leaves bfloat16 on L4 / A100. Attention is **SDPA** (the custom code already uses it). First run downloads ~1.2 GB `model.safetensors` plus ~1.3 GB `codec.pth`. Peak VRAM for a short clip should stay well under 8 GB.

**Not in this notebook:** SGLang Omni / OpenAI speech API, supervised fine-tuning, cloning a real person's voice, and the 12 languages outside the supported list that appear only in MiniMax eval.

**Links:** [model card](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air) · [code](https://github.com/Edge0-AI/Audio8_TTS) · [paper](https://arxiv.org/abs/2610.14776) · [playground](https://huggingface.co/spaces/Edge0/Audio8-TTS-Realistic-Air) · [announcement](https://x.com/SamuelZengML/status/2115881504102551856)


## 0. Early GPU check

Print `torch.cuda.is_available()`, the GPU name, and free VRAM before installing anything. Need a CUDA GPU (T4 is the free-Colab target). The restart after the install cell drops this state; section 2 prints it again and sets `DEVICE` / `DTYPE`.

In [ ]:
import shutil

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; this notebook will load float16.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install the card's Transformers stack (keep Colab's torch)

The [model card](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air#installation) install is:

```
pip install "torch>=2.5.0" "torchaudio>=2.5.0" \
  "transformers>=4.57.0,<5" "soundfile>=0.12" "safetensors>=0.4"
```

Smoke-tested with **PyTorch 2.8.0** and **Transformers 4.57.3**. Current Colab already satisfies `torch>=2.5.0` and `torchaudio>=2.5.0`. This cell installs the remaining pins and does **not** replace Colab's CUDA torch.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Do **not** install `flash-attn`. Do **not** install the SGLang Omni adapter.


In [ ]:
%pip install -q "transformers>=4.57.0,<5" "soundfile>=0.12" "safetensors>=0.4"

# Card: torch>=2.5.0, torchaudio>=2.5.0, transformers>=4.57.0,<5, soundfile>=0.12, safetensors>=0.4.
# Keep Colab's CUDA torch / torchaudio (already >= 2.5). Use %pip, not python -I.

import sys

import torch
import torchaudio

print("python", sys.version.split()[0])
print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
print("torchaudio (unchanged)", torchaudio.__version__)


> **Restart the runtime now:** *Runtime → Restart session*, then continue from the next cell.
>
> Colab has already imported its preinstalled `transformers` (often 5.x). The card pin is `transformers>=4.57.0,<5` and is invisible until the process restarts. You do **not** need to rerun the install cell after the restart unless it failed.


## 2. Runtime check and T4 dtype

Need a CUDA GPU. Print `torch.cuda.is_available()`, the device name, and free VRAM again (the restart dropped the first cell's state). Pick **float16** on a T4 and **bfloat16** only when the GPU has native BF16 (sm80+).


In [ ]:
import os
import shutil

import torch
import transformers
from packaging.version import Version

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

tf_ver = Version(transformers.__version__)
if tf_ver.major != 4 or tf_ver < Version("4.57.0"):
    raise SystemExit(
        f"transformers {transformers.__version__} is loaded; Audio8-TTS pins >=4.57.0,<5. "
        "Run the install cell, then Runtime → Restart session."
    )

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime → Change runtime type → T4 GPU, then rerun from the GPU-check cell.")

# T4 (sm_75) has no FlashAttention-2. Custom code uses SDPA; keep mem-efficient / math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

free_b, total_b = torch.cuda.mem_get_info()
major, minor = torch.cuda.get_device_capability()
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")

# Card CUDA default is bfloat16 (H20). T4 has no native BF16 — use float16 there.
NATIVE_BF16 = major >= 8
DEVICE = "cuda"
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
MODEL_ID = "Edge0/Audio8-TTS-Realistic-Air"
print("DEVICE", DEVICE, "| DTYPE", DTYPE)
if not NATIVE_BF16:
    print("No native bfloat16; loading float16 (not a bf16-only path).")


## 3. Load Audio8-TTS-Realistic-Air (`trust_remote_code`)

Same API as the [model card](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air#usage):

```python
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=dtype).eval().to(device)
```

Review the custom files in the Hub repo (`modeling_arktts.py`, `processing_arktts.py`, `modeling_arktts_codec.py`) before you `trust_remote_code`. The first run downloads `model.safetensors` (~1.2 GB), `codec.pth` (~1.3 GB), and the tokenizer. The codec is loaded lazily on the first `decode_audio` / `encode_audio` call.


In [ ]:
from transformers import AutoModel, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=DTYPE,
).eval().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("repo:", MODEL_ID)
print("class:", type(model).__name__)
print(f"parameters: {n_params:,} ({n_params / 1e6:.1f}M, card 601M excludes the codec)")
print("codec_sample_rate:", model.config.codec_sample_rate)
print("num_codebooks:", model.config.num_codebooks)
print("max_seq_len:", model.config.max_seq_len)
print("dtype:", next(model.parameters()).dtype, "| device:", next(model.parameters()).device)
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"VRAM after load: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )


## 4. Multilingual TTS without a reference

Generation with no `reference_audio` uses the card's default voice. First cell is the card snippet for English. Second cell adds Chinese (a Preview language the authors smoke-tested) through the same `processor` → `generate` → `decode_audio` path.

`max_new_tokens=1024` is the card cap (~47 s at 44.1 kHz / 2048 samples per frame). Short lines stop on EOS much earlier. Keep utterances short so a free T4 stays interactive.


In [ ]:
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

# Card example — generation without a reference.
inputs = processor(
    text=["Welcome to Audio8 TTS. It is wonderful to hear from you."],
    return_tensors="pt",
)
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)

audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
if audio.size == 0 or not np.isfinite(audio).all():
    raise SystemExit("English decode returned empty or non-finite audio.")

sr = int(model.config.codec_sample_rate)
sf.write("audio8-en.wav", audio, sr)
print(f"English → audio8-en.wav | {len(audio) / sr:.2f} s | {sr} Hz | codes {tuple(output.codes.shape)}")
display(Audio(audio, rate=sr))


Same path, Chinese. The processor takes a list of strings either way — one utterance here so playback stays obvious.


In [ ]:
sr = int(model.config.codec_sample_rate)
inputs = processor(
    text=["你好，欢迎使用 Audio8 语音合成。"],
    return_tensors="pt",
)
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)

audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
if audio.size == 0 or not np.isfinite(audio).all():
    raise SystemExit("Chinese decode returned empty or non-finite audio.")

sf.write("audio8-zh.wav", audio, sr)
print(f"Chinese → audio8-zh.wav | {len(audio) / sr:.2f} s | {sr} Hz | codes {tuple(output.codes.shape)}")
display(Audio(audio, rate=sr))


## 5. Zero-shot voice clone from a clip this notebook just made

`reference_audio` is a path (or array). `reference_text` must match the spoken content **exactly**. There is no public third-party clip here — the reference is a take this notebook synthesises first (no-reference generation), then clones. That keeps the audio license-clean and avoids cloning a real person.

Only clone a voice you have permission to use.


In [ ]:
sr = int(model.config.codec_sample_rate)
REF_TEXT = "Hi, this is my voice."
CLONE_TEXT = "Hello from a cloned take. Same speaker, new words."
ref_path = "audio8-clone-ref.wav"
clone_path = "audio8-clone.wav"

# 1) Build a short reference the notebook owns.
inputs = processor(text=[REF_TEXT], return_tensors="pt")
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)
ref_audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
if ref_audio.size == 0 or not np.isfinite(ref_audio).all():
    raise SystemExit("Reference decode returned empty or non-finite audio.")
sf.write(ref_path, ref_audio, sr)
print("reference (model-made, no real speaker):", REF_TEXT)
display(Audio(ref_path))

# 2) Card zero-shot clone: reference_audio + matching reference_text.
inputs = processor(
    text=[CLONE_TEXT],
    reference_audio=[ref_path],
    reference_text=[REF_TEXT],
    return_tensors="pt",
)
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)
clone_audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
if clone_audio.size == 0 or not np.isfinite(clone_audio).all():
    raise SystemExit("Clone decode returned empty or non-finite audio.")
sf.write(clone_path, clone_audio, sr)
print("clone from", ref_path, "→", CLONE_TEXT)
print(f"{clone_path} | {len(clone_audio) / sr:.2f} s | {sr} Hz")
display(Audio(clone_path))


## 6. Playback and save — a newly added language

Portuguese is one of the **12 languages Realistic Air added** on top of Preview. Same no-reference API, then write a 44.1 kHz WAV and play it. Swap the string for Arabic, Hindi, Russian, or any other supported language if you want a different sample.


In [ ]:
sr = int(model.config.codec_sample_rate)
PT_TEXT = "Olá, este é um exemplo de voz em português."
pt_path = "audio8-pt.wav"

inputs = processor(text=[PT_TEXT], return_tensors="pt")
inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)
pt_audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
if pt_audio.size == 0 or not np.isfinite(pt_audio).all():
    raise SystemExit("Portuguese decode returned empty or non-finite audio.")

sf.write(pt_path, pt_audio, sr)
print(f"Portuguese → {pt_path} | {len(pt_audio) / sr:.2f} s | {sr} Hz")
print("WAV written at", sr, "Hz — play above, or download", pt_path)
display(Audio(pt_path))


## 7. Caveats and cleanup

- **Licence.** Apache-2.0 on code and weights. Keep the [NOTICE](https://huggingface.co/Edge0/Audio8-TTS-Realistic-Air/blob/main/NOTICE) attribution (DualAR / Fish Audio S2 Pro).
- **Install.** Card pins: `torch>=2.5.0`, `torchaudio>=2.5.0`, `transformers>=4.57.0,<5`, `soundfile>=0.12`, `safetensors>=0.4`. Keep Colab's torch. Use `%pip`, not `python -I`. Restart after the install so the 4.57 pin is the loaded module.
- **API.** `AutoProcessor` / `AutoModel` with `trust_remote_code=True`. No-reference: `processor(text=[...], return_tensors="pt")`. Clone: add `reference_audio` and an exact `reference_text`. Then `model.generate(..., return_dict_in_generate=True)` and `model.decode_audio(output.codes)`. Sample rate is `model.config.codec_sample_rate` (44100).
- **T4 / dtype.** Card CUDA default is bfloat16. This notebook uses **float16** on sm75 (no native BF16). Do not force a bf16-only path on a free T4. If a waveform is non-finite, rerun the load cell with `DTYPE = torch.float32` (still fits ~15 GB).
- **Languages.** 23 supported; quality is not equal. Finnish, Thai, Vietnamese, and Cantonese are weaker on the card's tables. Split long passages — text, reference audio, and reference transcript share a 2048-position context.
- **Cloning.** Clean reference, transcript must match. This notebook clones a clip it just synthesised. Do not upload a recording of a real person without consent.
- **Not Silencio-2, not Maya 2, not Swar, not Ming-3.0, not Drama 3.** Those are different stacks and licences. This one is Transformers DualAR + a bundled 44.1 kHz codec.
- **Serving.** For batched GPU serving see the [SGLang Omni adapter](https://github.com/Edge0-AI/Audio8_TTS#sglang-omni-serving). Not installed here.

Free the GPU before you open another notebook:


In [ ]:
import gc

if torch.cuda.is_available():
    print(f"before cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

del model, processor, output, waveforms, waveform_lengths
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**3:.2f} GB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )
else:
    print("CPU runtime — nothing to empty on the GPU.")
